In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from pathlib import Path
#%run utils/utils_helper_funcs.ipynb

In [6]:
def ozone_concentration_ts(hist, aunz, bona, tena):
    # Parse dates
    start = str((hist['time'].values)[0])[:4]
    end = str((hist['time'].values)[len(hist['time'])-1])[:4]
    # Condense ozone data into 2D 
    ozone_to_plot = []
    times = []
    for ds in [hist, aunz, bona, tena]:
        ds_prep = ds.drop_vars('ilev')
        ds_sfc = ds_prep.sel(lev=max(ds.lev.values))
        ds_sfc = ds_sfc.resample(time='QS-DEC').mean(dim='time')
        ds_flat = ds_sfc.mean(dim=['lat','lon'])
        ozone = ds_flat.O3.values
        ozone = ozone * 1e9
        ozone_to_plot.append(ozone)
        times.append(ds_flat.time)

    labels = ['Global Run', 'AUNZ Mask', 'BONA Mask', 'TENA Mask']
    colors = ['maroon', 'orangered', 'darkorange', 'gold']
    
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    
    # Add data to figure window
    for i in range(len(ozone_to_plot)):
        plt.plot(times[i], ozone_to_plot[i], label=labels[i], c=colors[i])
    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    plt.legend(fontsize='large')
    plt.xlabel('Time', fontsize=16)
    plt.ylabel('Parts per Billion', fontsize=16)
    plt.title(f'Surface Ozone Concentration by Simulation, {start} - {end}', fontsize=16)

In [7]:
def pm_concentration_ts(hist, aunz, bona, tena):
    # Parse dates
    start = str((hist['time'].values)[0])[:4]
    end = str((hist['time'].values)[len(hist['time'])-1])[:4]
    pm_to_plot = []
    times = []
    for ds in [hist, aunz, bona, tena]:
        ds_prep = ds.drop_vars('ilev')
        ds_sfc = ds_prep.sel(lev=max(ds.lev.values))
        ds_sfc = ds_sfc.resample(time='QS-DEC').mean(dim='time')
        ds_flat = ds_sfc.mean(dim=['lat','lon'])
        pm = ((ds_flat.PM25.values)*1e9)/1.225
        pm_to_plot.append(pm)
        times.append(ds_flat.time)

    labels = ['Global Run', 'AUNZ Mask', 'BONA Mask', 'TENA Mask']
    colors = ['maroon', 'orangered', 'darkorange', 'gold']
    
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    
    # Add data to figure window
    for i in range(len(pm_to_plot)):
        plt.plot(times[i], pm_to_plot[i], label=labels[i], c=colors[i])
    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    plt.legend(fontsize='large')
    plt.xlabel('Time', fontsize=16)
    plt.ylabel('μg/m³', fontsize=16)
    plt.title('Surface PM₂₅ Concentration by Simulation, 2015 - 2020', fontsize=16)

In [10]:
def difference_ts(hist, aunz, bona, tena, aerosol, time_range=None):
     # Handle dates
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        hist = hist.sel(time=slice(start, end))
        aunz = aunz.sel(time=slice(start, end))
        bona = bona.sel(time=slice(start, end))
        tena = tena.sel(time=slive(start, end))
    else:
            start = str((hist['time'].values)[0])[:4]
            end = str((hist['time'].values)[len(hist['time'])-1])[:4]

    diffs = []
    times = []

    # Process historical data
    hist_prep = hist.drop_vars('ilev')
    hist_sfc = hist_prep.sel(lev=max(hist.lev.values))
    hist_sfc = hist_sfc.resample(time='QS-DEC').mean(dim='time')
    hist_flat = hist_sfc.mean(dim=['lat','lon'])

    for ds in [aunz, bona, tena]:
        ds_prep = ds.drop_vars('ilev')
        ds_sfc = ds_prep.sel(lev=max(ds.lev.values))
        ds_sfc = ds_sfc.resample(time='QS-DEC').mean(dim='time')
        ds_flat = ds_sfc.mean(dim=['lat','lon'])
        if aerosol=='ozone':
            difference = hist_flat.O3.values - ds_flat.O3.values
        elif aerosol=='PM25':
            difference = hist_flat.PM25.values - ds_flat.PM25.values
        difference = difference*1e9
        diffs.append(difference)
        times.append(ds_flat.time)

    labels = ['AUNZ Mask', 'BONA Mask', 'TENA Mask']
    colors = ['orangered', 'darkorange', 'gold']
    
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()

    for i in range(len(diffs)):
        plt.plot(times[i], diffs[i], label=labels[i], c=colors[i])
    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    a_title = aerosol.capitalize()
    plt.legend(fontsize='xx-large')
    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    plt.xlabel('Time', fontsize=16)
    plt.ylabel('Concentration Difference (ppb)', fontsize=16)
    plt.title(f'Surface Global {a_title} Difference by Simulation, {start} - {end}', fontsize=16)